# W tej części zajmujemy się załadowaniem i przetworzeniem danych do postaci użytecznej #


## 1. Konfiguracja Środowiska i Import Bibliotek ##

W celu przeprowadzenia analizy ilościowej, wykorzystujemy zestaw bibliotek dedykowanych do obliczeń naukowych, statystyki oraz wizualizacji danych finansowych.

### Stos Technologiczny:

*   **Przetwarzanie Danych:**
    *   `pandas`: Zarządzanie strukturami danych typu DataFrame, obsługa szeregów czasowych.
    *   `numpy`: Operacje wektorowe i funkcje matematyczne (np. logarytmowanie).
*   **Wizualizacja:**
    *   `matplotlib` & `seaborn`: Tworzenie technicznych wykresów cen, wolumenu oraz rozkładów statystycznych.
*   **Ekonometria i Statystyka:**
    *   `statsmodels.tsa`: Biblioteka do analizy szeregów czasowych (Testy stacjonarności ADF, analiza autokorelacji ACF/PACF).
    *   `scipy.stats`: Weryfikacja hipotez statystycznych, w tym testy normalności rozkładu (Jarque-Bera).
*   **Integracja Projektu:**
    *   `sys` & `os`: Konfiguracja ścieżek systemowych umożliwiająca importowanie autorskich modułów z katalogu `src/`.

---

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import os

# Narzędzia statystyczne
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from scipy import stats

# Dodanie folderu src do ścieżki (wyjście z notebooks/ do głównego katalogu)
sys.path.append(os.path.abspath('../'))
from src.data_loader import load_data, clean_financial_data, analyze_data_content

# Ustawienia estetyczne wykresów
%matplotlib inline
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = [12, 6]
plt.rcParams['figure.dpi'] = 100

## 2. Wczytywanie i czyszczenie danych ##

Ładowanie danych

In [16]:
from pathlib import Path

# To znajdzie główny folder projektu (BICC_2026), 
# idąc od miejsca, gdzie jest ten notebook o dwa poziomy w górę
BASE_DIR = Path(os.getcwd()).parent
path_daily = BASE_DIR / "data" / "raw" / "BICC_Daily_Markets_Anon.csv"
path_weekly = BASE_DIR / "data" / "raw" / "BICC_Weekly_Macro_Anon.csv"
path_monthly = BASE_DIR / "data" / "raw" / "BICC_Monthly_Macro_Anon.csv"


data_daily = load_data(str(path_daily))
data_weekly = load_data(str(path_weekly))
data_monthly = load_data(str(path_monthly))

✅ Ustawiono indeks czasowy na kolumnie: Date
✅ Ustawiono indeks czasowy na kolumnie: Date
✅ Ustawiono indeks czasowy na kolumnie: Date


Analiza zawartości danych dziennych

In [17]:

print(data_daily.info())

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 1629 entries, 2020-01-01 to 2026-03-25
Data columns (total 20 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Unnamed: 1                 1629 non-null   float64
 1   Unnamed: 2                 1629 non-null   int64  
 2   Unnamed: 3                 1628 non-null   float64
 3   Unnamed: 4                 1628 non-null   float64
 4   Unnamed: 5                 1628 non-null   float64
 5   Unnamed: 6                 1628 non-null   float64
 6   Unnamed: 7                 1628 non-null   float64
 7   Unnamed: 8                 1628 non-null   float64
 8   Unnamed: 9                 1628 non-null   float64
 9   Unnamed: 10                1628 non-null   float64
 10  Unnamed: 11                1627 non-null   float64
 11  Unnamed: 12                1627 non-null   float64
 12  Unnamed: 13                1628 non-null   float64
 13  Unnamed: 14                162

In [18]:
# zmiana nazw kolumn
daily_mapping = {
    'Unnamed: 1': 'Target_Asset_Close',
    'Unnamed: 2': 'Target_Asset_Volume',
    'Unnamed: 3': 'Equity_Index_RegionA_Close',
    'Unnamed: 4': 'Equity_Index_RegionA_Volume',
    'Unnamed: 5': 'Equity_Index_RegionB_Close',
    'Unnamed: 6': 'Equity_Index_RegionB_Volume',
    'Unnamed: 7': 'Bond_Yield_RegionA_Close',
    'Unnamed: 8': 'Bond_Yield_RegionA_Volume',
    'Unnamed: 9': 'Commodity_Energy_1_Close',
    'Unnamed: 10': 'Commodity_Energy_1_Volume',
    'Unnamed: 11': 'Safe_Haven_Metal_Close',
    'Unnamed: 12': 'Safe_Haven_Metal_Volume',
    'Unnamed: 13': 'Risk_Index_Close',
    'Unnamed: 14': 'Risk_Index_Volume'
}

# Zmiana nazw
data_daily.rename(columns=daily_mapping, inplace=True)

# Czyszczenie: wypełnianie luk w cenach ( Close ) zgodnie z instrukcją
close_cols = [col for col in data_daily.columns if '_Close' in col]

# Czyszczenie: zamiana NA w Volume na 0 (jeśli tak zdecydujecie)
volume_cols = [col for col in data_daily.columns if '_Volume' in col]

print(close_cols)
print(volume_cols)

['Target_Asset_Close', 'Equity_Index_RegionA_Close', 'Equity_Index_RegionB_Close', 'Bond_Yield_RegionA_Close', 'Commodity_Energy_1_Close', 'Safe_Haven_Metal_Close', 'Risk_Index_Close', 'Commodity_Energy_2_Close', 'Supply_Chain_Index_Close', 'ESG_Asset_Close']
['Target_Asset_Volume', 'Equity_Index_RegionA_Volume', 'Equity_Index_RegionB_Volume', 'Bond_Yield_RegionA_Volume', 'Commodity_Energy_1_Volume', 'Safe_Haven_Metal_Volume', 'Risk_Index_Volume', 'Commodity_Energy_2_Volume', 'Supply_Chain_Index_Volume', 'ESG_Asset_Volume']


In [19]:
# braki danych dla kolumn z cenami
data_daily[close_cols].isna().sum()

Target_Asset_Close            0
Equity_Index_RegionA_Close    1
Equity_Index_RegionB_Close    1
Bond_Yield_RegionA_Close      1
Commodity_Energy_1_Close      1
Safe_Haven_Metal_Close        2
Risk_Index_Close              1
Commodity_Energy_2_Close      1
Supply_Chain_Index_Close      1
ESG_Asset_Close               1
dtype: int64

In [20]:
# braki danych dla wolumenow
data_daily[volume_cols].isna().sum()

Target_Asset_Volume               0
Equity_Index_RegionA_Volume       1
Equity_Index_RegionB_Volume       1
Bond_Yield_RegionA_Volume         1
Commodity_Energy_1_Volume         1
Safe_Haven_Metal_Volume           2
Risk_Index_Volume                 1
Commodity_Energy_2_Volume      1629
Supply_Chain_Index_Volume      1629
ESG_Asset_Volume               1629
dtype: int64

In [21]:
data_daily.describe()

,Target_Asset_Close,Target_Asset_Volume,Equity_Index_RegionA_Close,Equity_Index_RegionA_Volume,Equity_Index_RegionB_Close,Equity_Index_RegionB_Volume,Bond_Yield_RegionA_Close,Bond_Yield_RegionA_Volume,Commodity_Energy_1_Close,Commodity_Energy_1_Volume,Safe_Haven_Metal_Close,Safe_Haven_Metal_Volume,Risk_Index_Close,Risk_Index_Volume,Commodity_Energy_2_Close,Commodity_Energy_2_Volume,Supply_Chain_Index_Close,Supply_Chain_Index_Volume,ESG_Asset_Close,ESG_Asset_Volume
count,1629.000000,1629.0,1628.000000,1628.0,1628.000000,1628.000000,1628.000000,1628.000000,1628.000000,1628.0,1627.000000,1.627000e+03,1628.000000,1.628000e+03,1628.000000,0.0,1628.000000,0.0,1628.000000,0.0
mean,1.114488,0.0,20.959920,0.0,2293.258535,4737.467445,73.963348,33429.675061,3.002158,0.0,4328.696863,3.141462e+07,15920.156550,5.727406e+09,49991.192875,NaN,10162.617322,NaN,6475.080467,NaN
std,0.057450,0.0,7.834496,0.0,813.141685,23519.499215,18.709723,18847.653236,1.400965,0.0,788.875401,1.527809e+07,4655.670317,2.004788e+09,45972.983045,NaN,23076.125054,NaN,2122.354283,NaN
min,0.959619,0.0,11.860000,0.0,1477.300049,0.000000,19.330000,0.000000,0.499000,0.0,2385.820068,0.000000e+00,6994.290039,2.169020e+09,3510.000000,NaN,12.000000,NaN,1612.000000,NaN
25%,1.076426,0.0,15.937500,0.0,1799.074951,98.000000,64.975002,23364.000000,1.560000,0.0,3753.560059,2.250825e+07,12321.190430,4.423320e+09,26423.750000,NaN,1385.750000,NaN,5449.250000,NaN
50%,1.102536,0.0,19.020000,0.0,1927.649963,317.500000,74.915001,30759.000000,3.684500,0.0,4224.870117,2.788970e+07,14940.169922,5.086435e+09,35191.000000,NaN,1866.000000,NaN,7018.000000,NaN
75%,1.166317,0.0,24.065000,0.0,2513.225037,865.500000,84.492498,39687.250000,4.214500,0.0,4947.514893,3.500830e+07,19490.150391,6.589732e+09,51250.000000,NaN,2484.250000,NaN,8088.000000,NaN
max,1.234111,0.0,82.690002,0.0,5318.399902,251274.000000,127.980003,235965.000000,4.988000,0.0,6173.319824,1.673299e+08,26119.849609,1.630873e+10,321415.000000,NaN,99700.000000,NaN,9747.000000,NaN


In [22]:
# usuniecie danych z pustymi wolumenami
data_daily = data_daily.drop(columns=['Target_Asset_Volume', 'Equity_Index_RegionA_Volume', 'Commodity_Energy_1_Volume', 'Commodity_Energy_2_Volume', 'Supply_Chain_Index_Volume', 'ESG_Asset_Volume'])

In [25]:
# uzupełnienie braków danych wcześniejszą wartościa
data_daily = data_daily.ffill()

## 3. Wstępna wizualizacja danych ##